# 📅 2026-06-29~30 개발 노트 : JWT 인증 연동 → 행동로그 user 연결 → 최근 본 게임## 🎯 오늘의 목표 (배포하려다 더 중요한 게 나옴)- [x] 마이페이지 토큰 만료 버그 수정- [x] **FastAPI ↔ Django JWT 인증 연동** (계획 외 핵심 인프라)- [x] 행동 로그(UserAction)에 user_id 연결- [x] 최근 본 게임 화면 (마이페이지)- [x] 원칙 정립 + 신작 데이터 방향 정리> 기준일은 체감 기준. 오늘은 "배포"로 시작했으나, 신작 데이터 논의 → user별 데이터 수집 인프라가 더 시급함을 깨닫고 그쪽으로 전환.

## 🛠 진행 상황 및 핵심 기록**1. 마이페이지 토큰 만료 버그 (출시 블로커)**- 증상: 어제 로그인 → 오늘 마이페이지 "정보를 불러오지 못했어요". Django 로그 `GET /api/auth/me/ 401`.- 원인: getMe()/submitOnboarding()이 **raw fetch**라 apiClient의 401 자동갱신 interceptor를 안 탐. access token(1h) 만료되면 그냥 실패.- 해결: 둘 다 **apiClient(axios)로 교체**. Django(8001) 절대 URL을 주면 baseURL(FastAPI 8000) 무시하고 그리로 감 + interceptor는 그대로 작동 → 401시 자동 refresh.- 교훈: Django 호출도 apiClient로 하면 자동갱신 공짜. raw fetch는 인증 흐름에서 피할 것.**2. ⭐ FastAPI ↔ Django JWT 인증 연동 (오늘의 핵심)**- 문제: 행동 로그 20개 전부 user=None. 로그인해도 "누가" 봤는지 안 박힘. → 개인화/설문의 재료가 안 모임.- 구조: 인증은 Django(SimpleJWT, HS256, SECRET_KEY 서명, 토큰에 user_id 클레임), 로그는 FastAPI가 받음. FastAPI가 Django JWT를 검증해야 함.- 작업:  - **시크릿 통일**(기반): docker-compose fastapi env에 `DJANGO_SECRET_KEY=${DJANGO_SECRET_KEY}` 추가. Django(66)·FastAPI(100) 둘 다 같은 .env 값. → 확인 `your-super-secret-dj...`.  - requirements.txt에 `PyJWT==2.10.1` + `--build` 재생성.  - config.py: `DJANGO_SECRET_KEY`, `JWT_ALGORITHM='HS256'`.  - **services/auth.py 신규**: `get_optional_user_id(request)` — Bearer 토큰 jwt.decode(SECRET_KEY, HS256) → user_id. **선택적 인증**: 토큰 없음/만료/위조 시 None (비로그인 익명 수집 유지가 핵심).  - routers/taste.py `record_action`: `user_id=get_optional_user_id(request)` 추출 → UserAction에 `user_id=` 저장. (모델엔 이미 user_id 컬럼+인덱스 있었음, "Phase 2에 채우기"로 미뤄둔 거였음.)- 실측: 로그인(user=2)으로 게임 클릭 → `user=2 detail_view app=1222670`, `user=2 rec_click` DB 저장 확인. detail_view도 user 박힘 = 프론트가 토큰 잘 보냄(apiClient interceptor), beacon 걱정 불필요.**3. 최근 본 게임 (마이페이지)**- Django `RecentGamesView`(`/api/auth/recent-games/`): 내 detail_view를 최신순, 중복 app_id 제거, 최근 12개 → games 조인(name/header_image/genres).- api.ts `getRecentGames()` (apiClient+Django 절대URL). 마이페이지에서 `Promise.all([getMe, getRecentGames])`.- 마이페이지 "최근 본 게임" 섹션: 게임 카드(이미지+이름, 클릭→상세), 빈 상태 "아직 본 게임이 없어요".- 실측: Stardew Valley / Snufkin / Sims 4 카드로 뜸. 클릭할수록 늘어남.

## 💡 핵심 원칙 (오늘 준태가 Claude를 교정)**"데이터 수집 인프라는 출시 전, 분석 로직은 데이터 쌓인 후"**- Claude 실수: "유저 없으니 설문/개인화는 출시 후"라고 뭉뚱그림.- 준태 교정: **데이터는 가만히 있으면 안 모인다. 수집 장치(인프라)를 먼저 만들어야** 출시 순간부터 쌓인다. JWT를 미리 깐 것과 동일 논리.- 올바른 구분:  - 수집 인프라 (행동로그 user연결 / 설문 UI+저장 / 찜) → **출시 전**  - 분석 로직 (행동→점수 개인화 / 설문응답→GPT점수 보정) → **데이터 쌓인 후**- 이 원칙이 오늘 전체를 관통. JWT 연동도 "최근 본 게임" 보려고가 아니라, **출시부터 user별 데이터 안 버려지게** 까는 거였음.**신작 데이터 (Project B) 방향 확정**- 4,190개 = GPT-5.4 Batch 고급 데이터 = **교사(teacher)**.- 신작 = 싼 모델 + few-shot(4,190개를 예시로) → knowledge distillation → 저비용 고품질. (models.py에 `fewshot_5.4based` 이미 설계됨.)- 기존 배치 파이프라인 7개 파일 있음(batch_generator/split/sender/processor/merger/db_updator). few-shot 변형 + 크롤링(신규)만 추가하면 됨.- **Project B = 별도 Claude 프로젝트/새 세션**. 이 채팅(추천서비스/배포)과 컨텍스트 분리. 적재 후 gem_percentile 전체 재계산 → score_v6 재검증이 A와의 접점.

## 📋 다음 할 일**출시 전 수집 인프라 (준태 원칙 — 미리 깔기, 단 하나씩):**```✅ 행동 로그 + user 연결       (완료)✅ 최근 본 게임 화면           (완료)⬜ 지표 검증 설문 수집 인프라  ← 준태가 강하게 원함. 다음 1순위.   - 게임 상세+스팀 갔다온 후 "이 게임 어땠어요?" 설문   - GPT 60지표 점수 vs 유저 체감 체크 → Community Validation(모트#3)   - 설계부터: 무엇을/어떻게 묻나(별점? 지표 슬라이더? 맞다/아니다?), 언제 뜨나   - JWT 인프라 덕에 user 연결은 이미 됨. 설문 UI+응답저장 테이블만.⬜ 회원정보 수정 (온보딩 재활용, 가벼움)⬜ 회원 탈퇴 (법적 필수 PIPA/GDPR)⬜ 찜하기+찜목록 (favorites가 로컬인지 DB인지 확인부터)⬜ 마이페이지 사이드바 정리 (위 항목들 5개+ 생기면 그릇으로)```**출시 후 (분석 로직 — 데이터 쌓인 후):**- 행동/설문 데이터 → score_v6에 user 레이어 (개인화)- 설문 응답 → GPT 지표 점수 보정**잔여:** requirements pytest 영구추가, pydantic Config→ConfigDict.**배포:** 위 수집 인프라 일단락 후. Vercel+Railway, pgvector 확인.**Project B:** 별도 세션 (few-shot 신작 파이프라인).

## 📌 환경 메모 (반복 참고)- **표준 기동:** `docker-compose up -d` + 별터미널 `cd frontend && npm run dev`- **재부팅 후 Exited:** 정상 — 그냥 `up -d` (down 불필요). Errno5 마운트깨짐과 구분.- **Errno 5 (마운트 깨짐):** `down && up -d` 완전재생성. 3회+면 Docker Desktop 재시작.- **라이브러리 추가 시:** 단순 restart 아니라 `up -d --build` (안 그럼 새 패키지 설치 안 됨).- **bash 시크릿 출력:** 값에 `!` 있으면 `event not found` — 작은따옴표(`'...'`)로 감쌀 것.- **JWT 연동 구조:** Django SECRET_KEY로 HS256 서명 → 같은 키 공유하면 FastAPI가 `jwt.decode`로 검증. 토큰에 user_id 클레임 있음. 선택적 인증으로 비로그인 익명 유지.- **인증 호출은 apiClient로:** raw fetch 쓰면 401 자동갱신 못 탐. Django 절대URL 줘도 interceptor 작동.*— 배포하려다 더 중요한 인프라(user별 데이터 수집)를 깔았다. 데이터는 가만히 안 모인다.*